# 01 — E2′ split generation: freeze `splits/p2_living.json` (living-out)

Thin one-shot runner: no logic here, only orchestration and display —
all logic lives in `sharp_har/` (inventory.py, splits.py). Ref. §2.2
(rotation), §0.1/§0.3 (freeze, per-rotation μ/σ), §10.3 (E2′, v5.2).

**What this session does:** rebuilds the day-1 inventory from the staged
data (reports go to a session scratch dir — the frozen day-1 reports are
never rewritten), re-runs the day-1 blocking gates, displays the
living-out contingency for inspection, then builds the rotation:
test = AR-6 (S6, living room), train pool = S1–S5 + S7, val = 15% of the
pool stratified by (AR-set, activity) with rare-cell pins, seed 42 — the
§2.2 mechanics plus the **2026-07-18 twin-binding amendment**
(`splits/CHANGELOG.md`): the dual-archive `*alt` twins are not
independent split units and follow their base trace's side, so a twin
pair can never straddle the train/val boundary (p2_lab already satisfies
this invariant by draw). `build_p2_rotation` is passed
`reference=splits/p2_lab.json`, so it aborts without writing anything if
the trace universe or the class/axes/window metadata differ from the
frozen primary rotation, or if μ/σ were not computed fresh (§0.3).

**Freeze semantics (§0.1):** this notebook runs ONCE. The JSON it writes
is committed and never regenerated or edited; a guard cell below aborts
if the split already exists. Archive the executed copy under
`notebooks/runs/` as `YYYY-MM-DD_s6out_split.ipynb`.


## Environment setup


In [ ]:
from pathlib import Path
import subprocess
import sys
import zipfile

REPO_URL = "https://github.com/FilippoIsoni/sharp-har.git"

# Locate or clone the repository root containing the sharp_har package.
# In Colab, the notebook may run from a temporary working directory.
cwd = Path.cwd().resolve()
if (cwd / "sharp_har").exists():
    REPO_DIR = cwd
elif (cwd.parent / "sharp_har").exists():
    REPO_DIR = cwd.parent
elif (cwd.parent.parent / "sharp_har").exists():
    REPO_DIR = cwd.parent.parent
else:
    REPO_DIR = Path("/content/sharp-har")
    if not REPO_DIR.exists():
        REPO_DIR.parent.mkdir(parents=True, exist_ok=True)
        subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)

# After the clone, so the file actually exists on a fresh runtime.
!pip install -q -r {REPO_DIR}/requirements.txt

sys.path.insert(0, str(REPO_DIR))

from google.colab import drive
from sharp_har.utils import get_git_hash, read_yaml, set_seed

paths_cfg = read_yaml(REPO_DIR / "configs" / "paths.yaml")
drive_root = Path(paths_cfg["drive_root"])
stage_dir = Path(paths_cfg["stage_dir"])

# Mount Drive unconditionally (idempotent).
drive.mount("/content/drive")

# Stage the zip archives if needed (same convention as 00_setup_smoke).
if not (stage_dir.exists() and any(stage_dir.rglob("*.txt"))):
    stage_dir.mkdir(parents=True, exist_ok=True)
    for zip_name in paths_cfg["zips"]:
        src = drive_root / zip_name
        dst = Path("/content") / zip_name
        print(f"copying {src} -> {dst}")
        subprocess.run(["cp", str(src), str(dst)], check=True)
        with zipfile.ZipFile(dst) as zf:
            zf.extractall(stage_dir)

set_seed(42)
print("Repo dir:", REPO_DIR)
print("git hash:", get_git_hash(REPO_DIR))
print("Stage dir:", stage_dir)


In [ ]:
# Freeze guard (§0.1): frozen splits are never regenerated. If this
# session must be redone for any reason, that is a team discussion, not
# a rerun.
SPLIT_PATH = REPO_DIR / "splits" / "p2_living.json"
assert not SPLIT_PATH.exists(), (
    f"{SPLIT_PATH} already exists - frozen splits are never regenerated (§0.1)."
)
print("freeze guard OK:", SPLIT_PATH.name, "does not exist yet")


## Day-1 inventory rebuild + blocking gates

Same functions and same gates as the day-1 session (axes, duplicate
resolution + safety net, AR-set/campaign coverage, NaN policy,
non-activity exclusion). All report outputs go to a **session scratch
directory**: the frozen day-1 `reports/*.csv` are read-only history and
are not rewritten by this session.


In [ ]:
from sharp_har.inventory import (
    build_inventory, resolve_duplicate_streams, assert_axes,
    assert_no_duplicate_files, assert_coverage, apply_nan_policy,
    exclude_non_activities, decide_classes,
)

SESSION_REPORTS = Path("/content/reports_s6out")  # frozen day-1 reports stay untouched
SESSION_REPORTS.mkdir(parents=True, exist_ok=True)

inventory_df = build_inventory(stage_dir, out_dir=SESSION_REPORTS)
inventory_df = resolve_duplicate_streams(inventory_df, out_dir=SESSION_REPORTS)

assert_axes(inventory_df)
assert_no_duplicate_files(inventory_df)
missing = assert_coverage(inventory_df)
assert not missing, f"missing AR-sets/campaigns: {missing} - blocker, do not freeze"

inventory_clean = apply_nan_policy(inventory_df, out_dir=SESSION_REPORTS)
inventory_clean = exclude_non_activities(inventory_clean, out_dir=SESSION_REPORTS)
classes_info = decide_classes(inventory_clean)
print(classes_info)


## Living-out contingency — inspect BEFORE freezing

§10.3: the new rotation's contingency is inspected and declared BEFORE
any launch. In this table the **AR-6 row is the test set**; every other
row is the train pool. Expected from the frozen day-1 contingency: every
(AR-set, activity) cell has 1–3 traces, i.e. **every cell is rare** and
the pin-1 mechanics of §2.2 dominate the val draw.


In [ ]:
from sharp_har.inventory import build_contingency_table

contingency = build_contingency_table(
    inventory_clean, SESSION_REPORTS / "contingency_s6out_session.csv"
)
print(contingency)


## Build + freeze `p2_living.json`

**Declared expectations** (local dry-run of 2026-07-18 on the frozen
trace universe, with the twin-binding amendment active — exact, because
the assignment depends only on the trace ids and seed 42, never on data
values; only μ/σ are computed here for real; the same dry-run also
reproduced the frozen p2_lab partition exactly under the pre-amendment
mechanics, validating the methodology): **train=80, val=6, test=15,
pinned=41**, and

- val = S1b_E, S1b_J2, S1c_S, S2a_R, S4a_C2, S4b_J1 (AR-1 ×3, AR-2 ×1,
  AR-4 ×2) — **no AR-3/5/6/7 in val** (AR-6 is the test; AR-7's single
  campaign is fully absorbed into train);
- val classes = {C, E, J, R, S}: **H, L and W absent** → the val
  macro-F1 of this rotation is a **5-class** number, not
  scale-comparable to the 8-class test macro-F1 — accepted per §2.2's
  explicit rare-cell clause, same caveat family as p2_lab's 5-class val
  (selection is within-run, so a k-class val metric stays valid);
- **both twin pairs land in train with their bases** (S4a_L+S4a_Lalt,
  S5a_L+S5a_Lalt) — the binding log lines from `build_p2_rotation`
  must confirm this;
- AR-7 (11 traces) entirely in train.

**If the printed sizes or the val list differ from the above, STOP and
investigate before committing anything** — it would mean the session's
inventory diverged from day 1 (and the reference universe check should
have caught it first).


In [ ]:
from sharp_har.inventory import trace_table
from sharp_har.splits import build_p2_rotation

p2_living = build_p2_rotation(
    inventory_clean,
    test_ar_set="AR-6",
    protocol="P2-living",
    out_path=SPLIT_PATH,
    labels=classes_info["labels"],
    reference=REPO_DIR / "splits" / "p2_lab.json",
)
print("train:", len(p2_living["train"]), "val:", len(p2_living["val"]),
      "test:", len(p2_living["test"]), "pinned:", len(p2_living["pinned_train_traces"]))
print("norm:", p2_living["norm"])

traces = trace_table(inventory_clean)
print("\nval composition:")
print(traces[traces["trace_id"].isin(p2_living["val"])].to_string(index=False))
print("\ntest composition:")
print(traces[traces["trace_id"].isin(p2_living["test"])].to_string(index=False))


## Final commit

`splits/p2_living.json` is **frozen once committed** (§0.1 — never
regenerated or edited):

```bash
cd sharp-har
git add splits/p2_living.json
git commit -m "E2': freeze the P2-living split (leave-S6-out)"
git push
```

Then archive the executed copy of this notebook under `notebooks/runs/`
as `YYYY-MM-DD_s6out_split.ipynb` and add the STATUS.md line (same
commit, per convention). Only AFTER the push, launch the training
runner `03_train_c1_ce_s6out.ipynb` — its readiness cell requires the
frozen split to be present in the clone.
